# 03 – Prompt variants

Can a better prompt fix the LLM's systematic errors from notebook 02?

Two prompt changes are compared with the zero-shot prompt on the same 200 test messages:

1. **Category descriptions:** a one-sentence description of every category.
2. **Few-shot:** 2 randomly sampled training messages per category.

The API runs are slow and use the free-tier token limit, so their results are saved to CSV files in `../data/`. The `run(...)` calls are commented out; everything below them reads the saved files.

## Setup

In [ ]:
import os, json, time
import pandas as pd
from dotenv import load_dotenv
from groq import Groq
from scipy.stats import binomtest

load_dotenv(override=True)
client = Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL = "openai/gpt-oss-120b"

test = pd.read_csv("../data/test_sample_200.csv")
labels = sorted(test["category"].unique())

# training data for few-shot examples (never use the test sample here)
TRAIN_URL = ("https://raw.githubusercontent.com/PolyAI-LDN/"
             "task-specific-datasets/master/banking_data/train.csv")
train = pd.read_csv(TRAIN_URL)
train = train[train["category"].isin(labels)]

## Prompt variants

The descriptions were written from the label definitions, not copied from test messages. The two transfer categories, which zero-shot confused most often, are described with the direction of the money (TO the customer vs FROM the customer).

In [ ]:
DESCRIPTIONS = {
    "balance_not_updated_after_bank_transfer": (
        "Money was sent TO the customer's own account (e.g. they transferred money in "
        "from another bank or account), but it has not appeared in their balance yet."
    ),
    "card_arrival": (
        "The customer ordered a new card and is asking when it will arrive or how to "
        "track the delivery. The card has not been delivered yet."
    ),
    "card_not_working": (
        "The customer already has the card, but it does not work in general "
        "(e.g. broken, not accepted anywhere, chip or contactless not working)."
    ),
    "card_payment_fee_charged": (
        "The customer was charged an unexpected extra fee for a card payment."
    ),
    "declined_card_payment": (
        "A specific card payment was declined or rejected and did not go through."
    ),
    "exchange_rate": (
        "Questions about currency exchange rates: what rate is used, why the rate "
        "looks wrong, or how currency conversion works."
    ),
    "lost_or_stolen_card": (
        "The customer's card that they already had is lost, missing or stolen, "
        "and they want to block or report it."
    ),
    "pending_card_payment": (
        "A card payment is still shown as pending and has not completed yet."
    ),
    "top_up_failed": (
        "The customer tried to add money (top up) to their account, but the top-up failed."
    ),
    "transfer_not_received_by_recipient": (
        "The customer sent money FROM their account TO another person or account, "
        "and the recipient says they have not received it."
    ),
}

In [ ]:
def examples_block(n_per_class=2, seed=42):
    """Pick n training examples per category for the few-shot prompt."""
    rows = []
    for lab in labels:
        sample = train[train["category"] == lab].sample(n_per_class, random_state=seed)
        for t in sample["text"]:
            rows.append(f'Message: {t}\nAnswer: {{"label": "{lab}"}}')
    return "\n\n".join(rows)

FEW_SHOT_EXAMPLES = examples_block()

# sanity check: no few-shot example may appear in the test sample
few_texts = {line[len("Message: "):].lower()
             for line in FEW_SHOT_EXAMPLES.split("\n") if line.startswith("Message: ")}
print("few-shot examples:", len(few_texts))
print("of which in the test sample:", test["text"].str.lower().isin(few_texts).sum())  # should be 0

In [ ]:
def build_prompt(text, variant):
    if variant == "descriptions":
        cats = "\n".join(f"- {lab}: {DESCRIPTIONS[lab]}" for lab in labels)
        extra = ""
    elif variant == "few_shot":
        cats = ", ".join(labels)
        extra = f"\nExamples:\n{FEW_SHOT_EXAMPLES}\n"
    else:  # zero_shot, same as notebook 02
        cats = ", ".join(labels)
        extra = ""
    return (
        "Classify the banking customer message into exactly one category.\n"
        f"Categories:\n{cats}\n{extra}\n"
        'Answer only with JSON: {"label": "<category>"}\n\n'
        f"Message: {text}"
    )

def classify(text, variant):
    start = time.time()
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": build_prompt(text, variant)}],
        response_format={"type": "json_object"},
        temperature=0,
        reasoning_effort="low",
    )
    seconds = time.time() - start
    try:
        label = json.loads(resp.choices[0].message.content)["label"]
    except (json.JSONDecodeError, KeyError, TypeError):
        label = "INVALID"
    if label not in labels:
        label = "INVALID"
    time.sleep(5)                       # longer prompts: stay under tokens-per-minute limit
    return label, seconds, resp.usage.prompt_tokens, resp.usage.completion_tokens

def run(variant):
    df = test.copy()
    df[["llm_label", "seconds", "in_tokens", "out_tokens"]] = [classify(t, variant) for t in df["text"]]
    df.to_csv(f"../data/llm_{variant}_results.csv", index=False)
    return df

## Evaluation functions

`summary` measures each variant on the full test set. `compare` counts the messages a variant fixed and broke relative to a baseline and runs McNemar's exact test on them (a binomial test on the messages where the two variants disagree).

In [ ]:
TRANSFER_PAIR = {"balance_not_updated_after_bank_transfer", "transfer_not_received_by_recipient"}

def load(variant):
    return pd.read_csv(f"../data/llm_{variant}_results.csv")

def summary(variant):
    df = load(variant)
    errors = df[df["llm_label"] != df["category"]]
    pair_errors = errors[errors["category"].isin(TRANSFER_PAIR) & errors["llm_label"].isin(TRANSFER_PAIR)]
    return {
        "variant": variant,
        "accuracy": round((df["llm_label"] == df["category"]).mean(), 3),
        "errors": len(errors),
        "transfer_pair_errors": len(pair_errors),
        "invalid": int((df["llm_label"] == "INVALID").sum()),
        "avg_tokens_per_text": round((df["in_tokens"] + df["out_tokens"]).mean()),
        "avg_seconds": round(df["seconds"].mean(), 2),
    }

def compare(base, other):
    """Fixed/broken messages vs a baseline + McNemar's exact test. Returns the new errors."""
    b, o = load(base), load(other)
    b_ok = b["llm_label"] == b["category"]
    o_ok = o["llm_label"] == o["category"]
    fixed = int((~b_ok & o_ok).sum())
    broken = int((b_ok & ~o_ok).sum())
    p = binomtest(fixed, fixed + broken, 0.5).pvalue if fixed + broken > 0 else 1.0
    print(f"{other} vs {base}: fixed {fixed}, broken {broken}, McNemar p = {p:.3f}")
    return o[b_ok & ~o_ok][["text", "category", "llm_label"]]

## Day 1: category descriptions

In [ ]:
# already run; results saved in ../data/llm_descriptions_results.csv
# desc = run("descriptions")

In [ ]:
compare("zero_shot", "descriptions")

## Day 2: few-shot (2 examples per category)

In [ ]:
# already run; results saved in ../data/llm_few_shot_results.csv
# few = run("few_shot")

In [ ]:
compare("zero_shot", "few_shot")

## Comparison

In [ ]:
pd.DataFrame([summary(v) for v in ["zero_shot", "descriptions", "few_shot"]])

## Findings

- **Descriptions** almost halved the targeted transfer-pair confusion (9 → 5), but fixed 4 messages and broke 3 others, so accuracy barely moved (0.930 → 0.935).
- **Few-shot** with random examples did not help: it fixed 3 and broke 4 messages, kept the transfer-pair confusion (8) and produced the first invalid answer, while using 2.6× more tokens than zero-shot.
- **No difference is statistically significant** (McNemar p = 1.0 in both cases).
- "Are you able to locate my card?" was mislabelled as `card_arrival` by both prompt variants: the message itself is ambiguous, not the prompt.
- More context does not guarantee better answers, and a fix aimed at one error type can move errors elsewhere. Every variant therefore has to be measured on the full test set against the same baseline.